## **Analysing Data with Pandas**

In [1]:
import pandas as pd

##### **1. Load the Clean Dataset**

In [2]:
df= pd.read_csv("datasets/kakuma_fresh_mart_sales_clean.csv", parse_dates=['date'])

In [3]:
df.head()

,receipt,date,branch,product,category,qty,price,payment,customer,imputed_price,is_return,unusual_qty,new_customer
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,False,True
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,False,True
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,False,True
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65.0,Cash,C0308,False,False,False,False
4,R10005,2026-10-01,Kakuma Town,Detergent,household,3.0,270.0,Cash,NaN,False,False,False,True


In [4]:
df.shape

(2467, 13)

In [5]:
# create revenue column
df['revenue']=  df['qty'] * df['price']

In [6]:
df.head(3)

,receipt,date,branch,product,category,qty,price,payment,customer,imputed_price,is_return,unusual_qty,new_customer,revenue
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,False,True,100.0
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,False,True,200.0
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,False,True,80.0


In [7]:
# remove unusual quantity (999,500, 250)

sales= df[df['unusual_qty']== False]

# Net Revenue
print("Net Revenue:", sales['revenue'].sum())

Net Revenue: 949214.0


##### **2. Where Does the Revenue Come From?**

> **Question 1**: Which `branch` brings in the most revenue, and is it because customers `spend more` or because there are `more of them`? 

In [8]:
# Revenue by branch
sales.groupby('branch')['revenue'].sum().sort_values(ascending=False)

branch
Kakuma Town    375795.0
Lodwar         287493.0
Kitale         285926.0
Name: revenue, dtype: float64

`Kakuma Town` has the highest revenue `(375795.0)`- `40%` compare to Lodwar `(287493.0)` and Kitale `(285926.0)`

In [9]:
# Branch summary - Revenue, Orders, AOV
branch_summary= sales.groupby('branch').agg(
    total_revenue= ('revenue', 'sum'),
    customers= ('receipt', 'count'),
    AOV= ('revenue', 'mean')
)

branch_summary.sort_values('total_revenue', ascending= False).round(2).reset_index()

,branch,total_revenue,customers,AOV
0,Kakuma Town,375795.0,1000,375.80
1,Lodwar,287493.0,727,395.45
2,Kitale,285926.0,737,387.96


> **Kakuma Town earns about 40% of revenue (KES 375,795 of 949,214),  because it sells to more customers, not because customers spend more.** It has 1,000 sales lines against about 730 elsewhere, and its average per line (KES 376) is the lowest 
of the three branches.

##### **3: What Drives Revenue?**

> **Question 2**: Which categories and products drive revenue?. Focus on share over the amount

In [10]:
# Revenue by category
cat_revenue= sales.groupby('category')['revenue'].sum().sort_values(ascending=False)

cat_revenue_pct= (cat_revenue/cat_revenue.sum()*100).round(1)

#Top 5 Categories(pct)
cat_revenue_pct.head(5)

category
grains       25.2
cooking      24.5
dairy        22.6
household    10.9
drinks       10.3
Name: revenue, dtype: float64

Three categories, `Grains`, `Cooking` and `Dairy`, account for about `72%` of revenue (25.2 + 24.5 + 22.6). Produce, the smallest, accounts for only 6.5 percent.


In [11]:
# Revenue by products
product_revenue= sales.groupby('product')['revenue'].sum().sort_values(ascending=False)

product_revenue_pct= product_revenue/product_revenue.sum()*100

#Top 5 Products
product_revenue_pct.round(1).head(5)

product
Eggs           15.0
Cooking oil    13.4
Rice            9.9
Wheat flour     8.5
Detergent       7.7
Name: revenue, dtype: float64

**Revenue vs Volume**

In [12]:
# Order by product
product_orders= sales.groupby('product')['qty'].sum().sort_values(ascending=False)

# Top 10
product_orders.head(10)

product
Water          552.0
Soda           505.0
Milk           500.0
Wheat flour    425.0
Sugar          413.0
Cooking oil    401.0
Maize flour    362.0
Rice           328.0
Soap           318.0
Eggs           298.0
Name: qty, dtype: float64

> **Revenue is concentrated: five of 19 products bring in 54% of revenue, and Grains,Cooking and Dairy together make up 72%**. Eggs is the top earner (about 15% of revenue) although it ranks only tenth by units sold, while Water is the top seller by units.

##### **4: Do Customers Behave Differently?**

> **Question 3:** Do returning customers spend differently from other shoppers, and how do
customers pay?

In [13]:
# new vs returning customers

sales['new_customer'].value_counts()

new_customer
True     1367
False    1097
Name: count, dtype: int64

In [14]:
# customer summary
customer_summary= sales.groupby('new_customer')['revenue'].agg(['count', 'mean','median'])

customer_summary.round(2).reset_index()


,new_customer,count,mean,median
0,False,1097,386.39,250.0
1,True,1367,384.31,260.0


> Returning average KES 386 per line and new customers KES 384. The medians are 250 and 260. These are practically the same. **There is no sign that loyalty members spend more per line.** 

**Payment Behaviour**

In [15]:
#payment methods in different branches
branch_payment= pd.crosstab(sales['branch'], sales['payment'], normalize='index')*100

branch_payment.round(1)

payment,Card,Cash,Credit,M-Pesa,Unknown
branch,,,,,
Kakuma Town,5.9,45.6,3.6,43.0,1.9
Kitale,7.7,46.3,2.6,40.3,3.1
Lodwar,7.3,41.7,3.6,44.8,2.6


In [16]:
sales.pivot_table(
    index='branch',
    columns=['payment'],
    aggfunc='size'
)

payment,Card,Cash,Credit,M-Pesa,Unknown
branch,,,,,
Kakuma Town,59,456,36,430,19
Kitale,57,341,19,297,23
Lodwar,53,303,26,326,19


> **Loyalty members do not spend more per sales line: KES 386 against 384 on average, and medians of 250 against 260, with over 1,000 lines in each group**. Payment habits are similar across branches: cash and M-Pesa each account for about 40 to 46% of
lines. 

##### **5: When Do We Sell Most?**
> **Question 4:** When do we sell most? 

**a. Daily Revenue**

In [17]:
#daily revenue
daily= sales.groupby(['date'])['revenue'].sum().reset_index()

# Get day names
daily['weekday'] = daily['date'].dt.day_name()

daily.head(10)

,date,revenue,weekday
0,2026-10-01,37362.0,Thursday
1,2026-10-02,42529.0,Friday
2,2026-10-03,55095.0,Saturday
3,2026-10-04,50932.0,Sunday
4,2026-10-05,43906.0,Monday
5,2026-10-06,46827.0,Tuesday
6,2026-10-07,44407.0,Wednesday
7,2026-10-08,37099.0,Thursday
8,2026-10-09,38116.0,Friday
9,2026-10-10,61031.0,Saturday


In [18]:
# Daily sales(7 days)
days_order= ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_sales= daily.groupby('weekday')['revenue'].mean().reindex(days_order)

daily_sales.reset_index().round(2)

,weekday,revenue
0,Monday,43471.00
1,Tuesday,43789.00
2,Wednesday,39637.00
3,Thursday,35973.67
4,Friday,39252.67
5,Saturday,56029.00
6,Sunday,58252.33


In [19]:
# Compare weekend vs week days avg revenue
daily['is_weekend']= daily['weekday'].isin(['Saturday', 'Sunday'])

daily.groupby('is_weekend')['revenue'].mean().reset_index().round(2)

,is_weekend,revenue
0,False,40424.67
1,True,57140.67


**Do weekends perform better in ALL branches**

In [22]:
sales['is_weekend'] = sales['date'].dt.day_of_week > 4

pvt= pd.pivot_table(
    sales,
    index='branch',
    columns= 'is_weekend',
    values='revenue',
    aggfunc='sum'
)

pvt.round(1)

C:\Users\USER\AppData\Local\Temp\ipykernel_6972\2392782340.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['is_weekend'] = sales['date'].dt.day_of_week > 4


is_weekend,False,True
branch,,
Kakuma Town,242209.0,133586.0
Kitale,182283.0,103643.0
Lodwar,181878.0,105615.0


In [23]:
days_per_group= sales.groupby('is_weekend')['date'].nunique()

per_day= pvt/ days_per_group

per_day.round(0)

is_weekend,False,True
branch,,
Kakuma Town,16147.0,22264.0
Kitale,12152.0,17274.0
Lodwar,12125.0,17602.0


> **Weekend days earn about 41% more than weekdays: KES 57,141 a day against 40,425 The lift is present in every branch (38% at Kakuma Town, 42% at Kitale, 45% at Lodwar).** 

> **So what:** schedule more staff and make sure stock is full for Saturday and Sunday.

> **Caveat:** the data covers only three weeks, with each weekday seen three times, so differences between individual weekdays are weak evidence

##### <u>**Findings summary**</u>

**Bottom line:** Sales are not weak at any one branch. The business is concentrated
in a few products and in the weekend, so staffing and stock should follow those.

1. Kakuma Town earns about 40% of revenue because of volume, not higher spending.
2. Five of 19 products bring in 54% of revenue. Eggs leads by revenue, not by units.
3. Loyalty members do not spend more per line, and payments are alike everywhere.
4. Weekend days earn about 41% more than weekdays, in every branch.

**Please confirm:** the three extreme quantities, and day-first slash dates.

**Not covered:** customers and visits, because the file records items only.